# Redrob Candidate Ranker Sandbox

Reproducible Google Colab demo for the Redrob candidate ranker. Clones the GitHub repo, downloads `candidates.jsonl` from Google Drive, runs the ranker offline on CPU, and validates the output CSV.

## 1. Install Dependencies

First, we install the CPU-only dependencies required for feature extraction and model inference. The model utilizes `numpy`, `scikit-learn`, and `xgboost` (falls back to LightGBM or Random Forest if needed).

In [ ]:
# Install required libraries
!pip install -q numpy scikit-learn xgboost pandas gdown

## 2. Clone Repository

Clone the ranker code from GitHub to get `rank.py`, `validate_submission.py`, and supporting files.

In [ ]:
import os

REPO_URL = "https://github.com/dhanasaur/Redrob__.git"
REPO_NAME = "Redrob__"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
else:
    print(f"{REPO_NAME} already exists, pulling changes...")
    %cd {REPO_NAME}
    !git pull

print(f"Current working directory: {os.getcwd()}")
print("Workspace files:", os.listdir("."))

## 3. Download Candidate Dataset

Download `candidates.jsonl` from Google Drive. The ranker streams this file line-by-line to keep memory usage low.

In [ ]:
import gdown

DRIVE_URL = "https://drive.google.com/file/d/1rY6g0LiWjPS2lxMWssi7F5QA_Zdq79Bq/view?usp=sharing"
CANDIDATES_PATH = "candidates.jsonl"

if not os.path.exists(CANDIDATES_PATH):
    print("Downloading candidates.jsonl from Google Drive...")
    gdown.download(DRIVE_URL, CANDIDATES_PATH, fuzzy=True, quiet=False)
else:
    print(f"{CANDIDATES_PATH} already exists, skipping download.")

if os.path.exists(CANDIDATES_PATH):
    size_mb = os.path.getsize(CANDIDATES_PATH) / (1024 * 1024)
    with open(CANDIDATES_PATH, "r", encoding="utf-8") as f:
        line_count = sum(1 for _ in f)
    print(f"Ready: {CANDIDATES_PATH} ({size_mb:.1f} MB, {line_count:,} candidates)")
else:
    print("ERROR: Download failed. Check that the Drive file is shared as 'Anyone with the link'.")

## 4. Run the Candidate Ranker

Next, we run the CPU-only offline ranker. It reads the candidates, extracts compact features, performs training/validation, and outputs a ranked CSV file with natural language reasoning for each candidate.

In [ ]:
# Run the ranker on the full candidate pool and write a top-100 submission CSV.
!python rank.py --candidates candidates.jsonl --out submission.csv --verbose

## 5. View Top Ranked Candidates

Let's display the top 10 candidates selected by the ranker model, including their model scores and the generated reasoning.

In [ ]:
import pandas as pd

if os.path.exists('submission.csv'):
    df = pd.read_csv('submission.csv')
    print(f"Ranked {len(df)} candidates. Showing Top 10:")
    pd.set_option('display.max_colwidth', None)
    display(df.head(10))
else:
    print("ERROR: submission.csv was not generated. Check the ranker output above.")

## 6. Validate Submission Format

Run the official validator to confirm the CSV has exactly 100 rows, correct columns, and monotonically decreasing scores.

In [ ]:
!python validate_submission.py submission.csv